# Connect to PostgreSQL

In [109]:
import psycopg2

try:
   # Connect to PostgreSQL
   connection = psycopg2.connect(
       dbname="Evolution",
       user="ev",
       password="Temp@123",
       host="192.168.4.51", # or your server's IP address
       port="5432" # default PostgreSQL port
   )
   print("Connection established successfully!")
except Exception as e:
   print(f"Error: {e}")


Connection established successfully!


In [110]:
cur = connection.cursor()

# Load the data

In [111]:
import pandas as pd
import os
import json


In [112]:
directory = "C:/Users/maryam.maksour/OneDrive - AL BAYARI/Desktop/RAG/DATA"


# Load the embedding model

In [113]:
from langchain_ollama import OllamaEmbeddings

emb = OllamaEmbeddings(model="bge-large", base_url="http://192.168.43.220:11435")

def generate_embedding(text): 
    text = str(text)
    embedding = emb.embed_query(text)
    return embedding

In [82]:
# Delete the table
x = cur.execute("""DROP TABLE """)

connection.commit()
connection.rollback()

In [114]:
connection.commit()
connection.rollback()

In [22]:
x = cur.execute("""ALTER TABLE Buildings ADD embed_address  VECTOR(1024);""")

connection.commit()
connection.rollback()

In [115]:
directory = "C:/Users/maryam.maksour/OneDrive - AL BAYARI/Desktop/RAG/DATA"

full_path = os.path.join(directory, "Projects.json")
file_name = "Projects"


In [116]:

if os.path.isfile(full_path):
    with open(full_path, 'r', encoding='utf-8-sig') as file:
                data = json.load(file)
                
                i = 0
                try:
                   for row in data[file_name]:
                           if 'Address' not in row or row['Address'] is None:
                                                         row['Address'] = "Null"
                           print(row['Address'])
                                                         
                           embedding = generate_embedding(row['Address'])
                           print(embedding)
                          
                           cur.execute("""UPDATE Projects SET embed_address = %s WHERE id = %s""",  
                              ( embedding, row['Id']))
                           
                           if (i+1) % 100 == 0:
                                 connection.commit()
                           i += 1
						 
                except Exception as e:
                     print(file_name)
                     print(e)
       
connection.commit()

Null
[-0.010386781, 0.0008649847, -0.01885093, 0.013304712, -0.021996602, -0.038682368, 0.022567026, 0.031833813, 0.0100040855, 0.04848672, 0.052370183, 0.016659271, -0.01448406, -0.009398523, -0.02515042, -0.0036572968, 0.009908231, 0.034320194, -0.054179166, -0.01817415, 0.01662718, 0.05094547, -0.08509635, -0.04354778, 0.025721766, 0.034609277, 0.022456862, 0.01098721, 0.03731453, 0.05125175, -0.022884572, -0.01623306, 0.0155764995, -0.030595072, 0.003039274, -0.046550497, 0.014800879, 0.020206096, 0.0062946198, -0.016661204, 0.008812381, -0.038645387, 0.008126198, -0.0033087202, -0.05969241, -0.00848698, -0.007726079, -0.049292184, -0.034368124, -0.030524228, -0.026759181, 0.002671956, 0.014380228, -0.010478277, 0.02154502, -0.053778753, 0.026834087, 0.037161816, -0.051652297, 0.011368143, 0.013142519, 0.004370375, 0.018950569, -0.05263419, 0.007241503, 0.020509925, -0.01831831, -0.019703124, 0.037676796, 0.0050549493, -0.020397805, -0.017914265, -0.0041950657, -0.04542186, -0.0064

In [39]:

from typing import List


def vector_to_literal(vec: List[float]) -> str:
    return "[" + ", ".join(str(float(x)) for x in vec) + "]"

 

In [97]:
s = "Business Bay"

In [98]:
y = generate_embedding(s)

In [99]:
m = vector_to_literal(y)


In [104]:
s = "SELECT buildings.id FROM buildings WHERE (buildings.embed_location <=> %s::vector < 0.35 OR buildings.embed_address <=> %s::vector < 0.35) AND buildings.status = 'Active'"

x = cur.execute(s, (m, m))


In [105]:
cur.fetchall()


[(34,)]